In [78]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import search_around_sky


In [125]:
def populate_sdss_fields(df):

    hdul = fits.open('data/dr16q_prop_May01_2024.fits')
    fits_data = hdul[1].data  # Assuming the data is in the first extension    
    fits_data2 = hdul[2].data  # Assuming the data is in the second extension

    
    # Add apparent_mag_i as a new field to fits_data2

    # Convert fits_data2 to Table, add column, convert back to FITS_rec
    table_data2 = Table(fits_data2)

    # Calculate apparent_mag_i from PSFFLUX (i-band is index 2)
    apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5
    table_data2['apparent_mag_i'] = apparent_mag_i

    fits_data2 = table_data2.as_array()

    fields1 = ['RA', 'DEC', 'SDSS_NAME', 'Z_SYS', 'LOGLBOL', 'LOGLBOL_ERR', 'LOGL3000', 'LOGL3000_ERR']
    fields2 = ['M_I', 'SN_MEDIAN_ALL', 'apparent_mag_i', 'PSFFLUX', 'PSFFLUX_IVAR', 'FUV', 'FUV_IVAR', 'NUV', 'NUV_IVAR', 'YFLUX', 'YFLUX_ERR', 'JFLUX', 'JFLUX_ERR', 'HFLUX', 'HFLUX_ERR', 'W1_FLUX', 'W1_FLUX_IVAR', 'W2_FLUX', 'W2_FLUX_IVAR', 'XMM_HARD_FLUX', 'XMM_HARD_FLUX_ERR']

    data = {}
    for f in fields1:
        data[f] = fits_data[f]
    for f in fields2:
        if np.array(fits_data2[f]).ndim > 1:
            print(np.shape(fits_data2[f]))
            data[f + '_u'] = fits_data2[f][:, 0]
            data[f + '_g'] = fits_data2[f][:, 1]
            data[f + '_r'] = fits_data2[f][:, 2]
            data[f + '_i'] = fits_data2[f][:, 3]
            data[f + '_z'] = fits_data2[f][:, 4]
        else:
            data[f] = fits_data2[f]

    df_fits = pd.DataFrame(data)
    df_fits['Z'] = df_fits['Z_SYS']

    # Calculate LOGLBOL for each row in df_fits based on Z_SYS
    df_fits['LOGLBOL_CALC'] = np.where(
        df_fits['Z'] < 0.7,
        np.log10(5.15) + df_fits['LOGL3000'],
        df_fits['LOGLBOL']
    )
    df_fits['LOGLBOL_ERR_CALC'] = np.where(
        df_fits['Z'] < 0.7,
        df_fits['LOGL3000_ERR'],
        df_fits['LOGLBOL_ERR']
    )

    df_fits['LOGLBOL'] = df_fits['LOGLBOL_CALC']
    df_fits['LOGLBOL_ERR'] = df_fits['LOGLBOL_ERR_CALC']

    cat = pd.read_parquet(f"data/S82/Catalog.parquet").set_index('idx')

    df_cat = cat.reset_index()[['objectId', 'RA', 'DEC']]
    df_cat = df_cat.rename(columns={'RA': 'RA_w', 'DEC': 'DEC_w'})

    # SkyCoord for both catalogs (assumes RA/DEC are in degrees)
    coords_cat  = SkyCoord(ra=df_cat['RA_w'].values * u.deg, dec=df_cat['DEC_w'].values * u.deg)
    coords_fits = SkyCoord(ra=df_fits['RA'].values    * u.deg, dec=df_fits['DEC'].values    * u.deg)

    # All matches within 1 arcsec
    idx_cat, idx_fits, sep2d, _ = search_around_sky(coords_cat, coords_fits, 1 * u.arcsec)

    # Give df_fits a key to merge on
    df_fits = df_fits.copy()
    df_fits['idx'] = np.arange(len(df_fits))

    # Take only the matched rows from df_cat (use iloc!), attach the matching df_fits index
    df_cat_match = df_cat.iloc[idx_cat].reset_index(drop=True).copy()
    df_cat_match['idx'] = idx_fits
    df_cat_match['sep_arcsec'] = sep2d.to(u.arcsec).value  # handy to keep

    # Inner join: repeats df_fits rows if there are multiple cat matches (expected behavior)
    df_catalog_merged = pd.merge(df_fits, df_cat_match, on='idx', how='inner')

    df_merged_final = pd.merge(df, df_catalog_merged, left_on='object_id', right_on='objectId', how='inner')

    return df_merged_final


In [126]:
df = pd.read_csv('data/sample_stone_fittedm2500.csv', dtype={'object_id': str})

df_populated = populate_sdss_fields(df)
df_populated.keys()

/var/folders/kx/qz91z8390zqfjrd9v6mt61w00000gn/T/ipykernel_8643/3003942283.py:14: RuntimeWarning: divide by zero encountered in log10
  apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5
/var/folders/kx/qz91z8390zqfjrd9v6mt61w00000gn/T/ipykernel_8643/3003942283.py:14: RuntimeWarning: invalid value encountered in log10
  apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5


(750414, 5)
(750414, 5)


Index(['object_id', 'sdss_name', 'apparent_mag_2500', 'apparent_mag_2500_err',
       'f_host_4200', 'alpha_lambda', 'alpha_lambda_err', 'redchi', 'RA_x',
       'DEC_x', 'SDSS_NAME_x', 'Z_SYS_x', 'LOGLBOL_x', 'LOGLBOL_ERR_x',
       'LOGL3000_x', 'LOGL3000_ERR_x', 'M_I_x', 'SN_MEDIAN_ALL_x',
       'apparent_mag_i_x', 'Z_x', 'LOGLBOL_CALC_x', 'LOGLBOL_ERR_CALC_x',
       'idx_x', 'objectId_x', 'RA_w_x', 'DEC_w_x', 'sep_arcsec_x', 'RA_y',
       'DEC_y', 'SDSS_NAME_y', 'Z_SYS_y', 'LOGLBOL_y', 'LOGLBOL_ERR_y',
       'LOGL3000_y', 'LOGL3000_ERR_y', 'M_I_y', 'SN_MEDIAN_ALL_y',
       'apparent_mag_i_y', 'PSFFLUX_u', 'PSFFLUX_g', 'PSFFLUX_r', 'PSFFLUX_i',
       'PSFFLUX_z', 'PSFFLUX_IVAR_u', 'PSFFLUX_IVAR_g', 'PSFFLUX_IVAR_r',
       'PSFFLUX_IVAR_i', 'PSFFLUX_IVAR_z', 'FUV', 'FUV_IVAR', 'NUV',
       'NUV_IVAR', 'YFLUX', 'YFLUX_ERR', 'JFLUX', 'JFLUX_ERR', 'HFLUX',
       'HFLUX_ERR', 'W1_FLUX', 'W1_FLUX_IVAR', 'W2_FLUX', 'W2_FLUX_IVAR',
       'XMM_HARD_FLUX', 'XMM_HARD_FLUX_ERR', 'Z_y'

In [127]:
df_populated

,object_id,sdss_name,apparent_mag_2500,apparent_mag_2500_err,f_host_4200,alpha_lambda,alpha_lambda_err,redchi,RA_x,DEC_x,...,XMM_HARD_FLUX,XMM_HARD_FLUX_ERR,Z_y,LOGLBOL_CALC_y,LOGLBOL_ERR_CALC_y,idx_y,objectId_y,RA_w_y,DEC_w_y,sep_arcsec_y
0,1391814,024028.11-005605.8,22.085476,0.000000,-99.000000,-0.984361,0.000000,0.386770,40.117132,-0.934958,...,-1.000000e+00,-1.000000e+00,1.029286,45.660053,0.003322,132633,1391814,40.117132,-0.934958,0.0
1,1391766,024037.61-005637.9,22.186022,0.005097,-99.000000,-0.558204,0.006879,0.904725,40.156739,-0.943867,...,-1.000000e+00,-1.000000e+00,1.834004,46.348279,0.004571,132689,1391766,40.156739,-0.943867,0.0
2,1391675,024051.10-010518.8,22.296194,0.009077,-99.000000,-1.285246,0.021992,0.495042,40.212922,-1.088575,...,-1.000000e+00,-1.000000e+00,1.840316,46.228666,0.003978,132789,1391675,40.212922,-1.088575,0.0
3,1391583,024105.06-005132.6,22.249751,0.006484,-99.000000,-1.200826,0.032955,0.489080,40.271103,-0.859071,...,-1.000000e+00,-1.000000e+00,1.786400,46.244913,0.005278,132886,1391583,40.271103,-0.859071,0.0
4,1391454,024126.71-004526.3,20.369019,0.000000,0.071462,-0.905262,0.000000,2.644420,40.361316,-0.757310,...,-1.000000e+00,-1.000000e+00,0.725465,45.945469,0.002820,133022,1391454,40.361316,-0.757310,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,1386822,025438.36+002132.6,22.250271,0.000381,-99.000000,-0.473399,0.000546,0.560327,43.659855,0.359071,...,-1.000000e+00,-1.000000e+00,2.464551,46.530852,0.002796,137969,1386822,43.659855,0.359071,0.0
185,1386720,025458.30-001142.1,21.705123,0.010952,0.047631,-0.821089,0.017449,0.258833,43.742942,-0.195039,...,3.793965e-15,2.613996e-15,0.923276,45.704447,0.008429,138075,1386720,43.742942,-0.195039,0.0
186,1386671,025510.55-000713.0,22.573746,0.007884,-99.000000,-1.686977,0.017552,0.462082,43.793983,-0.120288,...,7.651700e-15,2.914169e-15,1.693328,46.064567,0.004097,138125,1386671,43.793983,-0.120288,0.0
187,1386655,025513.02+000639.3,21.629531,0.010124,-99.000000,-1.248875,0.022826,0.345681,43.804291,0.110924,...,-1.000000e+00,-1.000000e+00,1.884356,46.555721,0.014218,138144,1386655,43.804291,0.110924,0.0


### Re-do matching (skip)

In [128]:
import numpy as np
import astropy.units as u
from astropy.table import Table, join, unique
from astroquery.xmatch import XMatch

# coords: SkyCoord of your sources
# data:   your original Table

# Build upload table with index
upload = Table()
upload['RA'] = coords.ra.deg
upload['DEC'] = coords.dec.deg
upload['src_idx'] = np.arange(len(coords))

def xmatch_nearest(upload_tbl, cat2, radius):
    t = XMatch.query(
        cat1=upload_tbl,
        cat2=f'vizier:{cat2}',
        max_distance=radius,
        colRA1='RA',
        colDec1='DEC',
    )
    t.sort(['src_idx', 'angDist'])
    t = unique(t, keys='src_idx', keep='first')
    return t

# Run matches
wise  = xmatch_nearest(upload, 'II/328/allwise', 1.5 * u.arcsec)
galex = xmatch_nearest(upload, 'II/312/ais',     1.5 * u.arcsec)

# Add src_idx to original data
data = data.copy()
data['src_idx'] = np.arange(len(data))

# Join results
combined = join(data, wise,  keys='src_idx', join_type='left')
combined = join(combined, galex, keys='src_idx', join_type='left')

#combined

### Write to CIGALE file

In [129]:
import numpy as np
from astropy.table import Table

def write_fluxes_to_cigale(table, path):
    """
    Write GALEX, SDSS, NIR, WISE, and XMM fluxes (µJy) to a CIGALE-compatible file.

    Expects columns (all fluxes in µJy):
      FUV, FUV_IVAR
      NUV, NUV_IVAR
      PSFFLUX_u, PSFFLUX_g, PSFFLUX_r, PSFFLUX_i, PSFFLUX_z
      PSFFLUX_IVAR_u, PSFFLUX_IVAR_g, PSFFLUX_IVAR_r, PSFFLUX_IVAR_i, PSFFLUX_IVAR_z
      YFLUX, YFLUX_ERR
      JFLUX, JFLUX_ERR
      HFLUX, HFLUX_ERR
      W1_FLUX, W1_FLUX_IVAR
      W2_FLUX, W2_FLUX_IVAR
      XMM_HARD_FLUX, XMM_HARD_FLUX_ERR
      redshift

    Missing or masked values are written as -99.
    """

    def clean(arr):
        """Replace masked or NaN with -99."""
        arr = np.asanyarray(arr)
        if np.ma.isMaskedArray(arr):
            arr = arr.filled(np.nan)
        return np.where(np.isfinite(arr), arr, -99.0)

    def ivar_to_err(flux, ivar):
        """Convert inverse variance to error, handling bad values."""
        flux = clean(flux)
        ivar = clean(ivar)
        return np.where((ivar > 0) & (flux != -99), 1.0 / np.sqrt(ivar), -99.0)

    out = Table()
    out['id']       = range(len(table))
    out['redshift'] = clean(table['Z_SYS_x'])

    # GALEX
    out['GALEX.FUV']     = clean(table['FUV'])
    out['GALEX.FUV.err'] = ivar_to_err(table['FUV'], table['FUV_IVAR'])
    out['GALEX.NUV']     = clean(table['NUV'])
    out['GALEX.NUV.err'] = ivar_to_err(table['NUV'], table['NUV_IVAR'])

    # SDSS (ugriz)
    for band in ['u', 'g', 'r', 'i', 'z']:
        out[f'SDSS.{band}']     = clean(table[f'PSFFLUX_{band}'])
        out[f'SDSS.{band}.err'] = ivar_to_err(table[f'PSFFLUX_{band}'],
                                              table[f'PSFFLUX_IVAR_{band}'])

    # VISTA NIR
    out['VISTA.Y']       = clean(table['YFLUX'])
    out['VISTA.Y.err']   = clean(table['YFLUX_ERR'])
    out['VISTA.J']       = clean(table['JFLUX'])
    out['VISTA.J.err']   = clean(table['JFLUX_ERR'])
    out['VISTA.H']       = clean(table['HFLUX'])
    out['VISTA.H.err']   = clean(table['HFLUX_ERR'])

    # WISE
    out['WISE.W1']       = clean(table['W1_FLUX'])
    out['WISE.W1.err']   = ivar_to_err(table['W1_FLUX'], table['W1_FLUX_IVAR'])
    out['WISE.W2']       = clean(table['W2_FLUX'])
    out['WISE.W2.err']   = ivar_to_err(table['W2_FLUX'], table['W2_FLUX_IVAR'])

    # XMM
    out['XMM.HARD']      = clean(table['XMM_HARD_FLUX'])
    out['XMM.HARD.err']  = clean(table['XMM_HARD_FLUX_ERR'])

    out.write(path, format='ascii.basic', overwrite=True)


In [131]:
write_fluxes_to_cigale(df_populated, "data/photometry_allbands.txt")

/var/folders/kx/qz91z8390zqfjrd9v6mt61w00000gn/T/ipykernel_8643/1039090401.py:35: RuntimeWarning: invalid value encountered in sqrt
  return np.where((ivar > 0) & (flux != -99), 1.0 / np.sqrt(ivar), -99.0)
/var/folders/kx/qz91z8390zqfjrd9v6mt61w00000gn/T/ipykernel_8643/1039090401.py:35: RuntimeWarning: divide by zero encountered in divide
  return np.where((ivar > 0) & (flux != -99), 1.0 / np.sqrt(ivar), -99.0)
